# Setup

Before start, please make sure you have all the dependencies installed.

In [ ]:
!pip install --quiet vizarr numba numcodecs

In [ ]:
import json
import numba
import numcodecs
import numpy as np

## Zoomable Mandelbrot Set

This notebook a contains `vizarr` example visualizing a generic multiscale zarr. The first cell contains code to create the underlying generatvie zarr store. It dynamically creates "chunks" at different zoom levels and associated array metadata.

In [ ]:

def create_meta_store(levels: int, tilesize: int, compressor, dtype: str):
    store = dict()
    store[".zgroup"] = json.dumps({"zarr_format": 2}).encode()

    datasets = [{"path": str(i)} for i in range(levels)]
    root_attrs = {"multiscales": [{"datasets": datasets, "version": "0.1"}]}
    store[".zattrs"] = json.dumps(root_attrs).encode()

    base_width = tilesize * 2 ** levels
    for level in range(levels):
        width = int(base_width / 2 ** level)
        meta = {
            "zarr_format": 2,
            "shape": [width, width],
            "chunks": [tilesize, tilesize],
            "dtype": np.dtype(dtype).str,
            "compressor": compressor.get_config() if compressor else None,
            "fill_value": 0,
            "order": "C",
            "filters": None,
        }
        store[f"{level}/.zarray"] = json.dumps(meta).encode()
    return store


@numba.njit
def mandelbrot(out, from_x, from_y, to_x, to_y, grid_size, maxiter):
    step_x = (to_x - from_x) / grid_size
    step_y = (to_y - from_y) / grid_size
    creal = from_x
    cimag = from_y
    for i in range(grid_size):
        cimag = from_y
        for j in range(grid_size):
            nreal = real = imag = n = 0
            for _ in range(maxiter):
                nreal = real * real - imag * imag + creal
                imag = 2 * real * imag + cimag
                real = nreal
                if real * real + imag * imag > 4.0:
                    break
                n += 1
            out[j * grid_size + i] = n
            cimag += step_y
        creal += step_x
    return out


@numba.njit
def tile_bounds(level, x, y, max_level, min_coord=-2.5, max_coord=2.5):
    max_width = max_coord - min_coord
    tile_width = max_width / 2 ** (max_level - level)
    from_x = min_coord + x * tile_width
    to_x = min_coord + (x + 1) * tile_width

    from_y = min_coord + y * tile_width
    to_y = min_coord + (y + 1) * tile_width

    return from_x, from_y, to_x, to_y


class MandlebrotStore:

    def __init__(self, levels, tilesize, maxiter=255, compressor=None):
        self.levels = levels
        self.tilesize = tilesize
        self.compressor = compressor
        self.dtype = np.dtype(np.uint8 if maxiter < 256 else np.uint16)
        self.maxiter = maxiter
        self._store = create_meta_store(levels, tilesize, compressor, self.dtype)

    def __contains__(self, key):
        if key in self._store:
            return True
        try:
            level, chunk_key = key.split("/")
            int(level)
            tuple(map(int, chunk_key.split(".")))
            return True
        except (ValueError, AttributeError):
            return False

    def __getitem__(self, key):
        if key in self._store:
            return self._store[key]

        try:
            # Try parsing pyramidal coords
            level, chunk_key = key.split("/")
            level = int(level)
            y, x = map(int, chunk_key.split("."))
        except:
            raise KeyError(key)

        from_x, from_y, to_x, to_y = tile_bounds(level, x, y, self.levels)
        out = np.zeros(self.tilesize * self.tilesize, dtype=self.dtype)
        tile = mandelbrot(out, from_x, from_y, to_x, to_y, self.tilesize, self.maxiter)
        tile = tile.reshape(self.tilesize, self.tilesize)

        if self.compressor:
            return self.compressor.encode(tile)

        return tile.tobytes()

### Running vizarr

Initalize the store implemented above, and open as a `zarr.Group` for vizarr. 

In [ ]:
# Initialize the store
store = MandlebrotStore(levels=50, tilesize=512, compressor=numcodecs.Blosc())

# This store implements the 'multiscales' zarr specification which is recognized by vizarr
# Pass the dict-like store directly to vizarr
grp = store

In [ ]:
import vizarr

viewer = vizarr.Viewer()
viewer.add_image(source=grp, name="mandelbrot")
viewer